<a href="https://colab.research.google.com/github/alicsrsustain-sudo/HVAC-Optimization-/blob/main/Motor_Run_Hours_Reduction_Calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Motor Run Hours Reduction Calculator
Calculates savings from reducing the number of hours a motor runs per year
(e.g. by adding occupancy controls, timers, or BMS scheduling).

Add, remove, or edit motors in the MOTORS list. Change cost_per_kwh to match your tariff.
"""

# ─────────────────────────────────────────────
#  INPUTS — change any value and re-run
# ─────────────────────────────────────────────

cost_per_kwh = 0.20   # € per kWh

# Each motor is a dict with:
#   name              - label for the output table
#   motor_size_kw     - rated motor size in kW
#   current_hours     - current annual running hours
#   proposed_hours    - proposed annual running hours after intervention
#   efficiency        - motor efficiency as a decimal (e.g. 0.9 = 90%)
#   speed             - speed as a fraction of full speed (e.g. 0.8 = 80%)
#   comments          - optional note on the intervention

motors = [
    {
        "name"           : "AHU - Sample",
        "motor_size_kw"  : 1,
        "current_hours"  : 3640,
        "proposed_hours" : 3120,
        "efficiency"     : 0.90,
        "speed"          : 0.80,
        "comments"       : "",
    },
]


# ─────────────────────────────────────────────
#  CALCULATIONS
# ─────────────────────────────────────────────

def calculate_motor(motor, cost_per_kwh):
    """
    Electrical input power = (motor_size × speed³) / efficiency   [Affinity Law]
    Annual cost = input power × hours × cost_per_kwh
    Saving comes purely from running fewer hours.
    """
    size     = motor["motor_size_kw"]
    eff      = motor["efficiency"]
    speed    = motor["speed"]
    hrs_cur  = motor["current_hours"]
    hrs_new  = motor["proposed_hours"]

    input_power  = (size * speed**3) / eff   # kW — same for both scenarios

    current_cost  = input_power * hrs_cur * cost_per_kwh   # €
    proposed_cost = input_power * hrs_new * cost_per_kwh   # €
    saving        = current_cost - proposed_cost            # €

    return {
        "Input Power (kW)"    : input_power,
        "Current Cost (€)"    : current_cost,
        "Proposed Cost (€)"   : proposed_cost,
        "Saving (€)"          : saving,
    }

results = [calculate_motor(m, cost_per_kwh) for m in motors]


# ─────────────────────────────────────────────
#  OUTPUT
# ─────────────────────────────────────────────

col_name = 20
col_num  = 9

print("=" * 100)
print("  MOTOR RUN HOURS REDUCTION CALCULATOR")
print("=" * 100)
print(f"  Cost per kWh: €{cost_per_kwh:.4f}")

print("\n" + "-" * 100)
print(
    f"  {'Motor':<{col_name}} "
    f"{'kW':>{col_num}} "
    f"{'Effic.':>{col_num}} "
    f"{'Speed':>{col_num}} "
    f"{'Cur Hrs':>{col_num}} "
    f"{'Prop Hrs':>{col_num}} "
    f"{'Cur Cost €':>{col_num+1}} "
    f"{'Prop Cost €':>{col_num+1}} "
    f"{'Saving €':>{col_num+1}}"
)
print("-" * 100)

total_current  = 0
total_proposed = 0
total_saving   = 0

for m, r in zip(motors, results):
    print(
        f"  {m['name']:<{col_name}} "
        f"{m['motor_size_kw']:>{col_num}.0f} "
        f"{m['efficiency']:>{col_num}.0%} "
        f"{m['speed']:>{col_num}.0%} "
        f"{m['current_hours']:>{col_num}.0f} "
        f"{m['proposed_hours']:>{col_num}.0f} "
        f"{r['Current Cost (€)']:>{col_num+1},.2f} "
        f"{r['Proposed Cost (€)']:>{col_num+1},.2f} "
        f"{r['Saving (€)']:>{col_num+1},.2f}"
        + (f"   # {m['comments']}" if m["comments"] else "")
    )
    total_current  += r["Current Cost (€)"]
    total_proposed += r["Proposed Cost (€)"]
    total_saving   += r["Saving (€)"]

print("-" * 100)
print(
    f"  {'TOTAL':<{col_name}} "
    f"{'':>{col_num}} "
    f"{'':>{col_num}} "
    f"{'':>{col_num}} "
    f"{'':>{col_num}} "
    f"{'':>{col_num}} "
    f"{total_current:>{col_num+1},.2f} "
    f"{total_proposed:>{col_num+1},.2f} "
    f"{total_saving:>{col_num+1},.2f}"
)
print("=" * 100)

pct_saving = (total_saving / total_current) * 100 if total_current else 0
print(f"\n  Overall Cost Saving: €{total_saving:,.2f}/year  ({pct_saving:.1f}%)")
print("=" * 100)
